In [74]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,udf,when
from pyspark.sql.types import StringType
import time

In [64]:
spark=SparkSession.builder.appName("sparkapp").getOrCreate()

In [65]:
df = spark.range(0, 100000).withColumn("value", col("id"))

In [33]:
df.take(10)

[Row(id=0, value=0),
 Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9)]

In [34]:
print("Before partitions:",df.rdd.getNumPartitions())

Before partitions: 2


In [35]:
df_repartitioned = df.repartition(10)

In [36]:
print("After partitions:", df_repartitioned.rdd.getNumPartitions())

After partitions: 10


In [37]:
df_coalesced=df_repartitioned.coalesce(2)

In [38]:
print("After coalesced:",df_coalesced.rdd.getNumPartitions())

After coalesced: 2


In [39]:
df_optimized=df.filter(col("value") >500).filter(col("id")<50000).select("id","value")

In [40]:
df_optimized.explain()

== Physical Plan ==
*(1) Project [id#29L, id#29L AS value#31L]
+- *(1) Filter ((id#29L > 500) AND (id#29L < 50000))
   +- *(1) Range (0, 100000, step=1, splits=2)




In [52]:
start_time =time.time()
count_uncached=df_optimized.count()
end_time=time.time()
print(f"count: {count_uncached} time: {round(end_time - start_time,4)} s")

count: 49499 time: 0.1314 s


In [53]:
df_optimized.cache()

DataFrame[id: bigint, value: bigint]

In [54]:
start_time =time.time()
count_first_cached=df_optimized.count()
end_time=time.time()
print(f"count: {count_first_cached} time: {round(end_time - start_time,4)} s")

count: 49499 time: 0.9451 s


In [55]:
start_time =time.time()
count_second_cached=df_optimized.count()
end_time=time.time()
print(f"count: {count_second_cached} time: {round(end_time - start_time,4)} s")

count: 49499 time: 0.34 s


In [56]:
start_time =time.time()
count_third_cached=df_optimized.count()
end_time=time.time()
print(f"count: {count_third_cached} time: {round(end_time - start_time,4)} s")

count: 49499 time: 0.2121 s


In [60]:
df_optimized.unpersist()

DataFrame[id: bigint, value: bigint]

In [67]:
data = [("alice", 30), ("akshya", 16)]
df = spark.createDataFrame(data, ["Name", "Age"])
df.show()

[Stage 45:>                                                         (0 + 1) / 1]

+------+---+
|  Name|Age|
+------+---+
| alice| 30|
|akshya| 16|
+------+---+



In [68]:
def can_drive(age):
    if age>=18:
        return "Can drive"
    return "Cannot drive"

In [70]:
age_categorize=udf(can_drive,StringType())

In [75]:
df_m = df.withColumn(
    "age_categorized",
    when(df.Age < 18, "Minor")
    .when(df.Age < 65, "Adult")
    .otherwise("Senior")
)

In [76]:
df_m.show()

+------+---+---------------+
|  Name|Age|age_categorized|
+------+---+---------------+
| alice| 30|          Adult|
|akshya| 16|          Minor|
+------+---+---------------+



In [78]:
spark.udf.register("sql_categorize",can_drive,StringType())

<function __main__.can_drive(age)>

In [79]:
df.craeteOrReplaceTempView("people")

AttributeError: 'DataFrame' object has no attribute 'craeteOrReplaceTempView'